# uxlens × LocateAnything — real CRO audit on a GPU

Run a **real** uxlens audit using NVIDIA's open-vocabulary vision model.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough).

> ⚠️ LocateAnything-3B is **non-commercial** (research/personal use only). uxlens itself is MIT.

## 1. Install uxlens + the browser + the model deps

In [ ]:
# uxlens with the model extra (torch/transformers/accelerate)
!pip install -q "uxlens[model] @ git+https://github.com/kannajune/uxlens.git"
# Headless Chromium for screenshots
!playwright install chromium
!playwright install-deps 2>/dev/null || true

## 2. Sanity check: GPU is available

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime to GPU!')

## 3. Audit your URL

First run downloads the model (~6 GB) — give it a few minutes.

In [ ]:
URL = "https://example.com"   # <-- put YOUR project URL here
VIEWPORT = "desktop"          # or "mobile"

from uxlens.audit import audit
from uxlens.report import print_summary, write_annotated_image

result = audit(URL, out_dir="out", viewport=VIEWPORT, backend="locate-anything")
print_summary(result)
write_annotated_image(result, "out/annotated.png")

## 4. See the annotated screenshot

In [ ]:
from IPython.display import Image, display
display(Image('out/annotated.png'))

## 5. (If boxes look wrong) inspect the raw model output

The parser expects `<box>x1 y1 x2 y2</box>` with coords in 0–1000. If a real
run differs, print the raw text below and share it — only `_parse_boxes` in
`uxlens/locator/locate_anything.py` needs adjusting.

In [ ]:
from PIL import Image as PILImage
from uxlens.locator import get_locator

loc = get_locator('locate-anything')
img = PILImage.open('out/screenshot.png').convert('RGB')
raw = loc._raw_inference(img, loc._prompt('primary call-to-action button'))
print(repr(raw))